# H&M 2년 M2 4모형 고속·자동재개 실행

한 번 실행하면 `m1`, `dual_clv_fixed`, `dual_shuffled_user`, `dual_adapter_only`를 순서대로 처리합니다. 각 epoch가 Drive에 저장되므로 런타임이 끊긴 뒤 다시 실행해도 마지막 완료 epoch부터 이어집니다. test와 holdout은 만들지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '303cca85686c09808d8a214a1f78ffdb03c774bd'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('검토된 코드:', REVIEWED_SHA)


In [ ]:
import json
import torch
from lightgcn_clv_dual_hm2y_suite import (
    MODELS, configure_hm2y_suite, preflight_summary,
    read_progress, run_hm2y_suite,
)

assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
cfg = configure_hm2y_suite(
    out_dir='/content/drive/MyDrive/논문/data/results_clv_dual_hm2y_suite',
    m1_checkpoint_dir='/content/drive/MyDrive/논문/data/results_v3_hm',
)
assert MODELS == ('m1', 'dual_clv_fixed', 'dual_shuffled_user', 'dual_adapter_only')
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


## 현재 상태만 확인

학습을 새로 시작하지 않고 Drive의 마지막 heartbeat를 읽습니다. 재접속 후 현재 단계와 epoch를 확인할 때 이 셀만 실행하세요.


In [ ]:
status = read_progress(cfg.out_dir)
print(json.dumps(status, ensure_ascii=False, indent=2))


## 네 모형 일괄 실행

아래 셀을 실행하면 완료된 단계는 건너뛰고, 중단된 학습은 마지막 epoch에서 자동 재개합니다. 별도의 승인 플래그는 없습니다.


In [ ]:
result_df = run_hm2y_suite(cfg)


In [ ]:
from IPython.display import display
import pandas as pd

display(result_df.sort_values(['model_id', 'lambda']))
print('최종 판정:', result_df.attrs['decision'])
print('결과 파일:', result_df.attrs['result_paths'])
display(pd.read_csv(result_df.attrs['result_paths']['delta_csv']))
